# Air-only validation

Validates the FEM solver against the Evert semi-analytical whole-space (air) reference.

```
master\
├── elfe3D_GPR\elfe3d_gpr
├── io\inputs\in_air_only\   ← generated here
└── examples\air\air.ipynb
```

In [ ]:
import sys, os, numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# ── locate master\ from examples\<name>\ (two levels up) ─────────────────
MASTER = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))

sys.path.insert(0, os.path.join(MASTER, 'io'))
sys.path.insert(0, os.path.join(MASTER, 'io', 'inputs'))
sys.path.insert(0, os.path.join(MASTER, 'io', 'outputs'))

# inputs
from survey import GPRSurvey

# runner
from runner import ProjectPaths, run_tetgen, run_solver

# outputs
from fieldreader import AnalyticalLoader, ElfeLoader, load_elfe_batch
from postprocess import field_error, all_errors, error_stats
from visualize   import (ReceiverLinePlot, ReceiverLineErrorPlot,
                          ReceiverLineCombined, ErrorHistogramPlot)


---
## Paths

In [ ]:
# ── set once per machine ───────────────────────────────────────────────────
paths = ProjectPaths(
    master_dir = MASTER,
    exec_rel   = r'elfe3D_GPR\elfe3d_gpr',
    use_wsl    = True,   # False if running from inside WSL
)

print('master   :', MASTER)
print('exec     :', paths.exec_path())


---
## 1 — Build and write inputs

`GPRSurvey.build()` assembles every sub-object, then `generate()` writes the `.poly` file and all FEM input files into `io/inputs/in_air_only/`.

In [ ]:
f    = 100e6
wave = 3e8 / f     # free-space wavelength = 3.0 m

# base_dir points at io\inputs\  — GPRSurvey creates in_air_only\ inside it
BASE_DIR = os.path.join(MASTER, 'io', 'inputs')

survey = GPRSurvey.build(
    experiment_name = 'air_only',
    base_dir        = BASE_DIR,

    # Domain
    x_e = [-wave/10, 1 + wave/10],
    y_e = [-wave/10,     wave/10],
    z_e = [-wave/10,     wave/10],

    # Materials — air only (whole-space)
    air_eps_r = 1.0,
    air_sigma = 1e-16,
    layer_thicknesses = [wave/10],  # thin dummy layer required by current implementation
    layer_eps_r       = [1.0],      # same as air
    layer_sigma       = [1e-16],
    layer_mu_r        = [1.0],
    layer_sigma_m     = [0.0],

    # Source
    ricker_central_f    = f,
    num_points_per_range = 1,
    antenna_position    = [0.0, 0.0, 0.025],
    source_type         = 6,
    current_direction   = 1,
    num_segments        = 1,
    s_f                 = 250,
    bh_f                = 1.0,
    box_present         = False,
    box_x               = [-1.0, 1.0],

    # Receivers  (inline = endfire/CO survey along x)
    num_receivers_inline  = 48,
    num_receivers_endfire = 0,
    num_receivers_oblique = 0,

    # Solver
    solver_type      = 2,
    max_ref_steps    = 0,
    max_unknowns     = 5_000_000,
    accuracy_tol     = 3e-5,
    output_fields_vtk = 1,

    # PML
    num_pml_layers      = 1,
    pml_layer_thickness = wave/10,
    pml_type            = 'lin',
    pml_decay_type      = 1,

    least_samples_per_wavelength = 20,
)

survey.generate()
print('poly :', survey.io.poly_file)
print('inputs:', survey.io.input_dir)


---
## 2 — Mesh with TetGen

In [ ]:
# TetGen reads the .poly written by survey.generate()
run_tetgen(paths, survey.io.poly_file)


---
## 3 — Run solver

In [ ]:
run_solver(paths)


---
## 4 — Load results

In [ ]:
result_txt = survey.io.output_dir / 'electric_fields_receiver_line.txt'
print('reading:', result_txt)

ef = ElfeLoader(
    filepath    = str(result_txt),
    label       = 'elfe3D  air_only',
    num_endfire = 48,   # matches num_receivers_inline above
).endfire()

print(f'r : {ef.r.min():.3f} – {ef.r.max():.3f} m   ({len(ef.r)} receivers)')


---
## 5 — Analytical reference

In [ ]:
# Half-space Evert quadrature reference (eps_r = 1 everywhere)
ANALYTICAL_DIR = r'C:\path\to\semi-analytic_100MHz'

evert = AnalyticalLoader(
    os.path.join(ANALYTICAL_DIR, 'Exx_single_freq_4_100MHz.csv'),
    label='Evert',
).endfire()


---
## 6 — Comparison plot

In [ ]:
ReceiverLinePlot(
    [evert, ef],
    suptitle='Endfire – Air only',
).plot()


## Error plot

In [ ]:
ReceiverLineErrorPlot(
    [ef], reference=evert,
    suptitle='Errors vs Evert – Air only',
).plot()


## Combined (fields + errors)

In [ ]:
ReceiverLineCombined(
    ef, evert,
    suptitle='Fields and errors – Air only',
).plot()


## Error histogram

In [ ]:
ErrorHistogramPlot(
    [ef], reference=evert,
    suptitle='Error distribution – Air only',
).plot()


## Printed error summary

In [ ]:
qty_names = ['Amplitude', 'Phase', 'Real', 'Imaginary']
print(f'\n── {ef.label} ──')
for qi, name in enumerate(qty_names):
    err = field_error(evert, ef, qi)
    m, s, mx = error_stats(err)
    scale, unit = (100, '%') if qi != 1 else (1, 'rad')
    print(f'  {name:12s}:  mean={m*scale:.3f}{unit}  '
          f'std={s*scale:.3f}{unit}  max={mx*scale:.3f}{unit}')
